# Project Setup

## Project Details (Add Project Name)


**Dataset:** Claude Generated Loan Data  
**Author:** Chuch  
**Date:** 2026-06-08  
**Objective:** playing around with pandas and completing challenges  
**Status:** In Progress / Complete

## Environment and Imports

In [ ]:
# ══════════════════════════════════════════════════════
# Environment Setup
# ══════════════════════════════════════════════════════

# ── Standard Library ──────────────────────────────────
import os
import sys
import warnings

# ── Data ──────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# ── Machine Learning ──────────────────────────────────
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ── Jupyter ───────────────────────────────────────────
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import display

def find_repo_root(start, repo_name='CreditRiskLearning'):
    path = os.path.abspath(start)
    while True:
        if os.path.basename(path) == repo_name:
            return path
        parent = os.path.dirname(path)
        if parent == path:
            raise FileNotFoundError(f"Could not find repo root: {repo_name}")
        path = parent

repo_root = find_repo_root(os.getcwd())
sys.path.append(os.path.join(repo_root, 'Core Resources'))
sys.path.append(os.path.join(repo_root, 'Core Resources', 'Scripts'))

import python_style_util as psu
import evaluation_utils as eu
import missingness_viz as mv        # missingess assessment functions

# ══════════════════════════════════════════════════════
# Display Settings
# ══════════════════════════════════════════════════════

%matplotlib inline

# Pandas
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.colheader_justify', 'left')



# --- Reproducibility ---
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Style Guide

In [ ]:
psu.show_tokens()
psu.show_fonts()


In [ ]:
import os

core = os.path.join(os.path.abspath('..'), 'Core Resources')
scripts = os.path.join(os.path.abspath('..'), 'Core Resources', 'Scripts')

print("Core Resources contents:")
print(os.listdir(core))

print("\nScripts contents:")
print(os.listdir(scripts) if os.path.exists(scripts) else "Scripts folder doesn't exist")

## Display Setttings

In [ ]:
# --- Pandas ---

pd.set_option('display.float_format', '{:.2f}'.format)  # 2 decimal places on floats
pd.set_option('display.max_columns', None)              # show all columns
pd.set_option('display.max_rows', 100)                  # show up to 100 rows
pd.set_option('display.width', None)                    # don't wrap wide dataframes
pd.set_option('display.colheader_justify', 'left')      # left-align column headers
pd.set_option('display.precision', 4)                   # decimal precision for describe()
pd.set_option('display.large_repr', 'truncate')         # truncate instead of summary for large dfs

# --- NumPy ---

np.set_printoptions(
    precision=4,        # decimal places (you have this)
    suppress=True,      # no scientific notation (you have this)
    linewidth=120,      # wrap width (you have this)
    threshold=1000,     # show up to 1000 elements before summarising with ...
    edgeitems=5,        # show 5 items at each end when it does summarise
)

# --- Matplotlib ---

%matplotlib inline
plt.rcParams['figure.dpi'] = 120             # sharper figures
plt.rcParams['savefig.dpi'] = 150            # higher res when saving
plt.rcParams['savefig.bbox'] = 'tight'       # no clipped labels when saving
plt.rcParams['savefig.facecolor'] = 'white'  # white background on saved figures

# --- Jupyter ---

InteractiveShell.ast_node_interactivity = 'all'  # print every expression, not just the last

# --- Warnings ---

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=ConvergenceWarning)

# --- Style ---

psu.set_style()

## Data Loading

In [ ]:
# Build path relative to notebook location
base_path = os.path.dirname(os.path.abspath('pandas_practice_notebook.ipynb'))
data_path = os.path.join(base_path)

# Load data
df_borrowers = pd.read_csv('borrowers.csv', sep=',')
df_credit = pd.read_csv('credit_bureau.csv', sep=',')
df_loans = pd.read_csv('loans.csv', sep=',')
df_repayments = pd.read_csv('repayments.csv', sep=',')


# Pandas Practice — Credit Risk Datasets

A task checklist to drill pandas end to end, using the four mock tables in `Practice Datasets/`: `borrowers.csv`, `loans.csv`, `repayments.csv`, `credit_bureau.csv`. They're deliberately messy (mixed casing, mixed date formats, duplicates, missing values, orphan keys, currency/percent strings) so cleaning tasks have real problems to solve — see the `README.md` alongside them for the full list of seeded issues.

**How to use this notebook:** write each answer from scratch in the empty code cell below its task — don't look up a reference solution first. This targets the recognition-vs-generation gap: you can already spot the right answer in a multiple choice list, the gap is producing the syntax cold. If you get stuck, note *why* before checking docs.

Sections roughly follow the study guide sequence, with extra reps on filtering/boolean indexing (requested focus) and on the specific weak spots from your last quiz: `.agg()` column-selector syntax, `pivot_table(aggfunc=...)`, the IQR outlier formula, multi-file `pd.concat()`, and data leakage identification.

## Section 0 — Setup

- [✅] 0.1 Print the `.shape` of each DataFrame to confirm they loaded.

In [ ]:
# Task 0.1

print('The borrowers dataset has', df_borrowers.shape[0], 'rows and', df_borrowers.shape[1], 'columns.')

print('The credit dataset has', df_credit.shape[0], 'rows and', df_credit.shape[1], 'columns.')

print('The loans dataset has', df_loans.shape[0], 'rows and', df_loans.shape[1], 'columns.')

print('The repayments dataset has', df_repayments.shape[0], 'rows and', df_repayments.shape[1], 'columns.')


## Section 1 — First Look

- [✅] 1.1 Run `.info()` on all four DataFrames. Note which columns have unexpected dtypes (e.g. numeric-looking columns stored as `object`).
- [✅] 1.2 Run `.describe()` on `loans` (numeric only), then again with `include='object'`.
- [✅] 1.3 Print `borrowers.shape`, `borrowers.columns`, `borrowers.dtypes`.
- [✅] 1.4 Run `.value_counts()` on `employment_status`, `loans['purpose']`, `loans['status']`, `repayments['payment_method']`, and `credit_bureau['bankruptcy_flag']`. Write down every distinct spelling/casing variant you find for each.
- [✅] 1.5 Compare `borrowers['borrower_id'].nunique()` to `len(borrowers)` — what does the gap tell you?
- [✅] 1.6 For each DataFrame, run `.isnull().sum().sort_values(ascending=False)` to rank columns by missingness.

In [ ]:
# Task 1.1
print('BORROWERS DATASET INFO\n')

df_borrowers.info()

# dtypes check out for each columns - not flagging anything

print('\n\n')

print('CREDIT DATASET INFO\n')

df_credit.info()

# total_debt as a string - need to look at this further

print('\n\n')

print('LOANS DATASET INFO\n')

df_loans.info()

# interest_rate as a string - also need to look at this further

print('\n\n')

print('REPAYMENTS DATASET INFO\n')

df_repayments.info()

# these all look fine

In [ ]:
# Task 1.2


print('LOANS DATASET describe (NUMERIC)\n')

df_loans.describe(include='number')

print('\n')

print('LOANS DATASET describe (OBJECT)\n')

df_loans.describe(include='object')

In [ ]:
# Task 1.3
print('Shape of the borrowers dataset: ', df_borrowers.shape, '\n\n')

print('List of columns in the borrowers dataset:\n\n', df_borrowers.columns, '\n\n')

print('List of datatypes in the borrowers dataset:\n ', df_borrowers.dtypes)


In [ ]:
# Task 1.4

df_borrowers['employment_status'].value_counts()

# distinct casing: 'employed', 'Employed', 'EMPLOYED', 'Unemployed', 'unemployed', 'self-employed', 'Self-employed', 'RETIRED', 'retired'

df_loans['purpose'].value_counts()

# distinct casing: 'home_improvement', 'Home Improvement', debt_consolidation, Debt Consolidation , Small Business, small_business, other, Other, Credit Card, credit_card

df_loans['status'].value_counts()

# distinct casing: Current, current, DEFAULT, Default, default, Paid Off, Paid off

df_repayments['payment_method'].value_counts()

# distinct casing: ACH, ach, check, Check, Credit Card, credit card, Wire, wire transfer

df_credit['bankruptcy_flag'].value_counts()

# distinct casing: N, No, Yes, Y

In [ ]:
# Task 1.5

df_borrowers['borrower_id'].nunique()

len(df_borrowers)

# there must be a duplicate borrower in there

df_borrowers['borrower_id'].value_counts()

# B003 in there twice

df_borrowers[df_borrowers['borrower_id']=='B003']

# Can confirm - "Robert Williams is in there twice"


In [ ]:
# Task 1.6

print('BORROWERS')

df_borrowers.isnull().sum().sort_values(ascending=False)

print('CREDIT')

df_credit.isnull().sum().sort_values(ascending=False)

print('LOANS')

df_loans.isnull().sum().sort_values(ascending=False)

print('REPAYMENTS')

df_repayments.isnull().sum().sort_values(ascending=False)


## Section 2 — Filtering & Boolean Indexing (extra reps)

- [ ] 2.1 Select `full_name` and `credit_score` from `borrowers` as a DataFrame (not two separate Series) — remember the double-bracket rule.
- [ ] 2.2 Build a boolean mask for `credit_score > 700`, print the mask on its own first, then use it to filter `borrowers`.
- [ ] 2.3 Do the same filter again using `.loc[]` syntax explicitly.
- [ ] 2.4 Filter `loans` for `loan_amount > 20000` **and** `term_months == 36`. Remember: `&`, not `and`, and each condition wrapped in parentheses.
- [ ] 2.5 Filter `loans` for `purpose` equal to `'other'` **or** `'small_business'` using `|`, then rewrite the same filter using `.isin()`.
- [ ] 2.6 Replicate task 2.4 using `df.query()` instead of boolean masks.
- [ ] 2.7 Filter `repayments` down to the problem rows: `amount_paid` is negative **or** missing. (Hint: you'll need `.isnull()` combined with a comparison.)
- [ ] 2.8 Take the `credit_score > 700` filtered subset from 2.2 and add a new column `high_score_flag = 1` to it using `.loc[]` correctly, so it does **not** raise a `SettingWithCopyWarning`.
- [ ] 2.9 Use `.str.contains()` to find every row in `borrowers` where `state` contains `'New'` (should catch `'New York'` regardless of what else is in the column).
- [ ] 2.10 Select the first 10 rows and first 3 columns of `loans` two ways: once with `.iloc[]`, once with `.loc[]`. Confirm both give the same result.
- [ ] 2.11 Filter `borrowers` for implausible ages: `age < 0` or `age > 100`. How many rows does this catch?
- [ ] 2.12 Filter `loans` to only the rows belonging to borrowers from `'TX'` or `'CA'` — first collect the matching `borrower_id`s from `borrowers` using `.isin()` on `state`, then filter `loans` using `.isin()` on that list of ids.

In [ ]:
# Task 2.1


In [ ]:
# Task 2.2


In [ ]:
# Task 2.3


In [ ]:
# Task 2.4


In [ ]:
# Task 2.5


In [ ]:
# Task 2.6


In [ ]:
# Task 2.7


In [ ]:
# Task 2.8


In [ ]:
# Task 2.9


In [ ]:
# Task 2.10


In [ ]:
# Task 2.11


In [ ]:
# Task 2.12


## Section 3 — Cleaning Messy Data

These map directly onto the issues seeded in the README.

- [ ] 3.1 Strip leading/trailing whitespace from `full_name` and convert to consistent title case.
- [ ] 3.2 Standardize `employment_status` down to exactly four values: `Employed`, `Unemployed`, `Self-Employed`, `Retired` — regardless of the casing in the raw data.
- [ ] 3.3 Standardize `state` to two-letter abbreviations only (map full names like `'California'` → `'CA'`).
- [ ] 3.4 Find and drop the exact duplicate borrower row (`B003`).
- [ ] 3.5 Decide how to handle the invalid ages found in 2.11 (cap, null out, or drop) and implement it. Write one sentence justifying your choice.
- [ ] 3.6 Flag any `credit_score` outside the valid 300–850 range and set those values to `NaN`.
- [ ] 3.7 Parse `loans['issue_date']` and `repayments['payment_date']` — both have at least four different date formats mixed in the same column — into proper `datetime64` columns.
- [ ] 3.8 Clean `interest_rate`: strip the `%` sign where present, then cast the whole column to `float`.
- [ ] 3.9 Handle the negative `loan_amount` value(s) — decide and implement a treatment, with a one-sentence justification.
- [ ] 3.10 Standardize `purpose` and `status` in `loans` to consistent casing/spacing (e.g. one canonical spelling of `debt_consolidation`, `Current`, `Default`, etc).
- [ ] 3.11 Standardize `payment_method` in `repayments` the same way.
- [ ] 3.12 Clean `total_debt` in `credit_bureau`: strip `$` and `,`, then cast to `float`.
- [ ] 3.13 Normalize `bankruptcy_flag` to a proper boolean column (`Y`/`Yes`/`1` → `True`, `N`/`No`/`0` → `False`).
- [ ] 3.14 Find and handle the duplicate repayment on loan `L1001` (same period billed twice).

In [ ]:
# Task 3.1


In [ ]:
# Task 3.2


In [ ]:
# Task 3.3


In [ ]:
# Task 3.4


In [ ]:
# Task 3.5


In [ ]:
# Task 3.6


In [ ]:
# Task 3.7


In [ ]:
# Task 3.8


In [ ]:
# Task 3.9


In [ ]:
# Task 3.10


In [ ]:
# Task 3.11


In [ ]:
# Task 3.12


In [ ]:
# Task 3.13


In [ ]:
# Task 3.14


## Section 4 — String Operations (`.str` accessor)

- [ ] 4.1 Clean a column in one chained line using `.str.strip().str.lower()`.
- [ ] 4.2 Strip `$` and `,` from `total_debt` in a single `.str.replace()` call (regex) rather than two separate calls.
- [ ] 4.3 Use `.str.contains()` to find every loan whose `purpose` mentions `'business'`.
- [ ] 4.4 Split `full_name` into `first_name` and `last_name` columns using `.str.split(...)`.

In [ ]:
# Task 4.1


In [ ]:
# Task 4.2


In [ ]:
# Task 4.3


In [ ]:
# Task 4.4


## Section 5 — Missing Data

- [ ] 5.1 For `annual_income`, compare two treatments: `dropna()` on that column vs `fillna()` with the median. State which you'd pick and why — and flag in a comment where the leakage risk would come from if this were done before a train/validation split.
- [ ] 5.2 Use `dropna(thresh=...)` to drop any `borrowers` row missing more than 2 fields. Remember `thresh` is a literal count, not a proportion.
- [ ] 5.3 Before imputing `annual_income`, create an `income_missing` indicator column (1/0) — do this **before** you fill the nulls, not after.
- [ ] 5.4 Fill missing `employment_status` values with `'Unknown'` rather than the most common category. Write one sentence on why that's usually the safer default for a categorical.

In [ ]:
# Task 5.1


In [ ]:
# Task 5.2


In [ ]:
# Task 5.3


In [ ]:
# Task 5.4


## Section 6 — New Columns, `apply()`, `map()`

- [ ] 6.1 After merging `credit_bureau` onto `borrowers` (see Section 9), create a vectorised ratio column `debt_to_income = total_debt / annual_income`.
- [ ] 6.2 Use `.map()` with a dict to convert `purpose` codes into nicer display labels (e.g. `'debt_consolidation'` → `'Debt Consolidation'`).
- [ ] 6.3 Use `.apply(axis=1)` to build a `risk_flag` column that is `1` if `credit_score < 620` **and** `status == 'Default'`, else `0`.
- [ ] 6.4 Rewrite 6.3 as a fully vectorised boolean expression (no `apply`). Compare readability and note which one you'd actually ship.

In [ ]:
# Task 6.1


In [ ]:
# Task 6.2


In [ ]:
# Task 6.3


In [ ]:
# Task 6.4


## Section 7 — Grouping & Aggregating

- [ ] 7.1 `groupby('employment_status')['credit_score'].mean()` — average credit score by employment status.
- [ ] 7.2 In a **single** `.agg()` call, compute `mean`, `sum`, and `count` of `loan_amount`, plus `mean` of `interest_rate`, grouped by `purpose`. This is the exact syntax that tripped you up last quiz — get the column-selector-to-function-list mapping right.
- [ ] 7.3 `groupby('status')['loan_amount'].sum()` — total exposure by loan status.
- [ ] 7.4 Group by `['state', 'purpose']` and compute mean `loan_amount`, then call `.reset_index()` on the result.
- [ ] 7.5 Create a boolean `is_default` column (`status == 'Default'`), then compute `groupby('purpose')['is_default'].mean()` for the true default *rate*, and separately `.sum()` for the raw *count*. Write one sentence on why someone might mistake the `.sum()` output for a rate.

In [ ]:
# Task 7.1


In [ ]:
# Task 7.2


In [ ]:
# Task 7.3


In [ ]:
# Task 7.4


In [ ]:
# Task 7.5


## Section 8 — Pivot Tables

- [ ] 8.1 `pivot_table(values='loan_amount', index='state', columns='purpose', aggfunc='mean')`. Remember: `aggfunc=`, not `agg=`.
- [ ] 8.2 Rebuild 8.1 with `aggfunc=['mean', 'count']` to get both in one table.
- [ ] 8.3 Reproduce the same result as 8.1 using `groupby(['state', 'purpose'])['loan_amount'].mean().unstack()`, and confirm the two approaches match.

In [ ]:
# Task 8.1


In [ ]:
# Task 8.2


In [ ]:
# Task 8.3


## Section 9 — Merging & Concatenating

- [ ] 9.1 `merge(borrowers, loans, on='borrower_id', how='inner')`. Compare the row count to `len(loans)` — which loan(s) got dropped, and why?
- [ ] 9.2 Redo 9.1 with `how='left'` (loans as the left table) so every loan is kept. Inspect the borrower fields for the orphan loan (`B999`) — what do they look like now?
- [ ] 9.3 Before merging `repayments` onto loans, aggregate it first: `groupby('loan_id')` → total `amount_paid` and count of payments. Then merge that summary onto your loans+borrowers table from 9.2.
- [ ] 9.4 `credit_bureau` has two report dates for some borrowers. Keep only the most recent report per `borrower_id` (hint: `sort_values` + `drop_duplicates(subset=..., keep='last')`, or `groupby` + `idxmax`), then merge that onto `borrowers`.
- [ ] 9.5 Split `credit_bureau` into two DataFrames by `report_date` (the `2023-01-01` batch and the `2023-07-01` batch), then use `pd.concat([...], ignore_index=True)` to put it back together and confirm you get the original row count back. This is deliberate practice on the exact `concat` syntax (`ignore_index=True`, not `reset_index=True`) flagged as a gap.

In [ ]:
# Task 9.1


In [ ]:
# Task 9.2


In [ ]:
# Task 9.3


In [ ]:
# Task 9.4


In [ ]:
# Task 9.5


## Section 10 — Sorting & Index Management

- [ ] 10.1 Sort `borrowers` by `credit_score` descending.
- [ ] 10.2 Sort `loans` by `state` ascending, then `loan_amount` descending, in one call.
- [ ] 10.3 Take the result of task 7.4 (grouped, not yet reset) and call `.reset_index()` on it. Then try `.reset_index(drop=True)` on a filtered DataFrame instead — explain the difference between the two in a comment.
- [ ] 10.4 After filtering `loans` in task 2.4, call `.sort_index()` on the result to restore original row order.

In [ ]:
# Task 10.1


In [ ]:
# Task 10.2


In [ ]:
# Task 10.3


In [ ]:
# Task 10.4


## Section 11 — Outlier Detection (IQR)

- [ ] 11.1 Compute `Q1`, `Q3`, and `IQR` for `annual_income` (cleaned). Compute `lower_bound = Q1 - 1.5 * IQR` and `upper_bound = Q3 + 1.5 * IQR`. Write out the formula from memory before checking it.
- [ ] 11.2 Use the bounds from 11.1 to filter out (or flag) outlier rows in `borrowers`.
- [ ] 11.3 Repeat the full IQR process for `loan_amount` in `loans`.

In [ ]:
# Task 11.1


In [ ]:
# Task 11.2


In [ ]:
# Task 11.3


## Section 12 — Applied Credit Risk Mini-Tasks

Pulls everything above together into the kind of analysis you'd actually do on Home Credit / German Credit.

- [ ] 12.1 Compute default rate (`status == 'Default'`) by `purpose`, and separately by `state`, each sorted highest to lowest.
- [ ] 12.2 For each borrower, compute total amount paid and total number of payments across all their loans (loan-level → borrower-level rollup, two aggregation steps).
- [ ] 12.3 Create a boolean `had_late_payment` feature per borrower: `True` if any of their payments had `days_late > 0`.
- [ ] 12.4 Compute average `days_late` by `payment_method` — does one method correlate with lateness?
- [ ] 12.5 Find borrowers who have **both** a `bankruptcy_flag` of `True` **and** at least one loan with `status == 'Default'` — a multi-table filter requiring a merge first.
- [ ] 12.6 Build a single tidy borrower-level master table joining all four sources with your cleaned columns and engineered features (`debt_to_income`, `had_late_payment`, `risk_flag`, etc). One row per borrower.

In [ ]:
# Task 12.1


In [ ]:
# Task 12.2


In [ ]:
# Task 12.3


In [ ]:
# Task 12.4


In [ ]:
# Task 12.5


In [ ]:
# Task 12.6


## Section 13 — Data Leakage Checkpoint (no code, just answer in this cell)

Go back through Sections 3, 5, 6, and 12. For each of the following, write one to two sentences on whether computing it on the **full** dataset (before a train/validation split) would leak information, and why:

1. The median used to fill missing `annual_income` (Task 5.1).
2. The IQR bounds used to flag outliers (Task 11.1).
3. The categories discovered for `pd.get_dummies()` if you were to one-hot encode `purpose` or `state`.
4. The per-`purpose` default rate computed in Task 12.1, if it were then used as a feature (target encoding).
5. The debt-to-income ratio in Task 6.1 — is this one actually safe? Why does it differ from the other four?

*(Answer here — double-click to edit this cell.)*